# Download data from bucket/git repo to speedup demo setup
This notebook generate a fake dataset for fine tuning on Databricks Documentation using DBRX to generate a question and answer for each documentation page.

We pre-ran this notebook for you and saved the results in our dataset bucket. We'll download it directly to accelerate the demo dataset preparation.

In [0]:
%run ./00-setup

In [0]:
folder = volume_folder

#1 Data collection and preparation

In this notebook, we'll download Databricks Documentation and use this as our Dataset.

To generate our Training & Testing Dataset containing question and answer for each documentation section, we'll be using Databricks `ai_query` and ask the DBRX Instruct LLM to ask a question and answer it (make sure you check the LLM license used by your SQL AI function).

*Note: For model fine-tuning, this is typically something you'd instead do manually, using real/human questions and answers. We will instead generate them to simplify our demo.*

In [0]:
%pip install beautifulsoup4 tiktoken==0.7.0 lxml==4.9.3 transformers==4.30.2 langchain==0.1.5

## Extracting databricks documentation sitemap & pages

Let's parse docs.databricks.com website and download the html content.

In [0]:
import requests
import xml.etree.ElementTree as ET

# Fetch the XML content from sitemap
response = requests.get("https://docs.databricks.com/en/doc-sitemap.xml")
root = ET.fromstring(response.content)
# Find all 'loc' elements (URLs) in the XML
urls = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
print(f"{len(urls)} Databricks documentation pages found")

#Let's split to 200 documentation page to make this demo faster:
#urls = urls[:200]

In [0]:
import requests
import pandas as pd
import concurrent.futures
from bs4 import BeautifulSoup
import re

# Function to fetch HTML content for a given URL
def fetch_html(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.content
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None

# Function to process a URL and extract text from the specified div
def process_url(url):
    html_content = fetch_html(url)
    if html_content:
        soup = BeautifulSoup(html_content, "html.parser")
        article_div = soup.find("div", itemprop="articleBody")
        if article_div:
            article_text = str(article_div)
            return {"url": url, "text": article_text.strip()}
    return None

# Use a ThreadPoolExecutor with 10 workers
with concurrent.futures.ThreadPoolExecutor(max_workers=50) as executor:
    results = list(executor.map(process_url, urls))

# Filter out None values (URLs that couldn't be fetched or didn't have the specified div)
valid_results = [result for result in results if result is not None]

#Save the content in a raw table
# Assuming 'valid_results' is a DataFrame that you want to save
spark.createDataFrame(valid_results) \
    .write \
    .mode('overwrite') \
    .saveAsTable('raw_documentation')

# Display the first 2 rows of the overwritten table
display(spark.table('raw_documentation').limit(2))

In [0]:
%sql
select * FROM raw_documentation

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_documentation  (id BIGINT GENERATED BY DEFAULT AS IDENTITY, url STRING, content STRING, title STRING);

### Splitting documentation pages in small chunks
LLM models typically have a maximum window size and you won't be able to compute embbeddings for too long texts.

In addition, the bigger your context is, the longer your inference will be.

Data preparation is key for your model to perform well and multiple strategy exist depending of your dataset:

- Split document in small chunks (paragraph, h2...)
- Truncate documents to a fix number
- It could sometime make sense to split in big chunks ans ask a model to summurize each chunks as a one-off job for faster live inference.

The number of token depends of your model. LLMs are shipped with a Tokenizer that you can use to count how many tokens will be created for a given sequence (usually > number of words) (see [hugging face documentation](https://huggingface.co/docs/transformers/main/tokenizer_summary) or [open AI](https://github.com/openai/tiktoken))


Make sure the tokenizer and context size limit you'll be using here matches your embedding model. Let's try an exemple with GPT3.5 tokenizer: `tiktoken.encoding_for_model("gpt-3.5-turbo")`

### Splitting our big documentation page in smaller chunks (h2 sections)

In this demo, we have a few big documentation article, too big for our models. We'll split these articles between HTML h2 chunks, and ensure that each chunk isn't bigger than 4000 tokens.<br/>
To do so, we'll be using the `tiktoken` librairy to count gtp3.5 tokens. 

Let's also remove the HTML tags to send plain text to our model.

In [0]:
from langchain.text_splitter import HTMLHeaderTextSplitter, RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, OpenAIGPTTokenizer

max_chunk_size = 1500

tokenizer = OpenAIGPTTokenizer.from_pretrained("openai-gpt")
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=max_chunk_size, chunk_overlap=50)
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=[("h2", "header2")])

# Split on H2, but merge small h2 chunks together to avoid too small. 
def split_html_on_h2(html, min_chunk_size = 20, max_chunk_size=1500):
  if not html:
      return []
  h2_chunks = html_splitter.split_text(html)
  chunks = []
  previous_chunk = ""
  # Merge chunks together to add text before h2 and avoid too small docs.
  for c in h2_chunks:
    # Concat the h2 (note: we could remove the previous chunk to avoid duplicate h2)
    content = c.metadata.get('header2', "") + "\n" + c.page_content
    if len(tokenizer.encode(previous_chunk + content)) <= max_chunk_size*0.8:
        previous_chunk += content + "\n"
    else:
        chunks.extend(text_splitter.split_text(previous_chunk.strip()))
        previous_chunk = content + "\n"
  if previous_chunk:
      chunks.extend(text_splitter.split_text(previous_chunk.strip()))
  # Discard too small chunks
  return [c for c in chunks if len(tokenizer.encode(c)) > min_chunk_size]
 
# Let's try our chunking function
html = spark.table("raw_documentation").limit(1).collect()[0]['text']
split_html_on_h2(html)

Let's now split our entire dataset using this functio and a pandas UDF.

We will also extract the title from the page (based on h1)

In [0]:
import pyspark.sql.functions as F

# Let's create a user-defined function (UDF) to chunk all our documents with spark
@F.pandas_udf("array<string>")
def parse_and_split(docs: pd.Series) -> pd.Series:
    return docs.apply(split_html_on_h2)
    
(spark.table("raw_documentation")
      .filter('text is not null')
      .withColumn('content', F.explode(parse_and_split('text')))
      .drop("text")
      .write.mode('overwrite').saveAsTable("databricks_documentation"))

display(spark.table("databricks_documentation"))

In [0]:
%sql 
SELECT * FROM databricks_documentation

## Let's use Databricks AI_GENERATE_TEXT to generate Questions for each documentation page  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS training_dataset_question (id BIGINT GENERATED ALWAYS AS IDENTITY, doc_id BIGINT, question STRING);
CREATE TABLE IF NOT EXISTS training_dataset_answer   (id BIGINT GENERATED ALWAYS AS IDENTITY, question_id BIGINT, answer STRING);

In [0]:
%sql
CREATE OR REPLACE FUNCTION GENERATE_QUESTION_FROM_DOC(doc STRING)
RETURNS STRING
RETURN
    ai_query(
        "databricks-dbrx-instruct",
        CONCAT(
            "Here is a documentation page from Databricks: ", doc,
            "\nWrite one questions a Databricks user (developer, data engineer, SQL Analyst, data scientist or devops) might ask themselves ",
            "which can be answered from the above documentation.",
            "\nGenerate one questions that can be answered as a user would ask. Just generate the question, no additional text."
        )
    )

In [0]:
import pyspark.sql.functions as F

spark.table('databricks_documentation').repartition(100) \
     .withColumn("question", F.expr("GENERATE_QUESTION_FROM_DOC(content)")) \
     .selectExpr("id as doc_id", "question") \
     .write.mode('overwrite').saveAsTable('training_dataset_question')

In [0]:
%sql
select * from training_dataset_question

## Generate the Answers based on the doc & the questions

In [0]:
%sql
CREATE OR REPLACE FUNCTION GENERATE_ANSWER_FROM_DOC(doc STRING, question STRING)
RETURNS STRING
RETURN 
    ai_query(
        "databricks-dbrx-instruct",
        CONCAT(
            "You are an assistant for python, spark, data engineering, setup/install and data science on Databricks. Here is a documentation page with potential relevant informations: \n\n ", doc,
            "\n\nUsing this information, answer the following question: \n\n",
            question
        )
)

In [0]:
from pyspark.sql import functions as F

df_q = spark.table('training_dataset_question').alias('q')
df_d = spark.table('databricks_documentation').alias('d')
question_df = df_q.join(df_d, df_q['q.doc_id'] == df_d['d.id'])

(question_df.repartition(100)
            .withColumn("answer", F.expr("GENERATE_ANSWER_FROM_DOC(content, question)"))
            .selectExpr("q.id as question_id", "answer")
            .write.mode('overwrite').saveAsTable('training_dataset_answer'))

In [0]:
%sql select * from training_dataset_answer order by question_id desc

In [0]:
%sql
SELECT * FROM databricks_documentation d
  INNER JOIN training_dataset_question q on q.doc_id = d.id
  INNER JOIN training_dataset_answer   a on a.question_id = q.id 

# Create tickets/support dataset from the doc

In [0]:
# Define the prompt with escaped quotes
prompt = """Using the provided text where HTML tags are represented in plain text format (e.g., <H1> is represented as H1), generate three dummy customer issues, each reflecting one of these specific intents: "impacting the prod," "urgent," or "not urgent." For each issue, generate a unique and random email ID for the customer submitting the request. Format the response as a JSON array, where each object includes the following three fields:

description - Provides a detailed account of the customer issue, reflecting realistic scenarios that could arise from the content described in the text. Make the wording of the issue to highlight indirectly the priority of the issue to be either "Urgent," "Impacting the Prod," or "Not Urgent". Write the description as it is coming from customer directly.
email - The fabricated email address of the customer submitting the issue. Make it unique coming from fictitious diffrent company domains.
priority - The designated priority level of the issue, categorized as "Urgent," "Impacting the Prod," or "Not Urgent.
created_on - The date on which issue was created. Format as "MM/DD/YYYY" (e.g., 01/01/2022). Assign random dates to issues based on the following ranges: 01/01/2023 - 01/01/2024,
\\n"""

In [0]:
# Concatenate the prompt with the doc column
from pyspark.sql.functions import concat, lit, col
df = spark.table("databricks_documentation")
df_with_prompt = df.withColumn("prompt", concat(lit(prompt), col("content")))
display(df_with_prompt)

In [0]:
%sql 
CREATE OR REPLACE FUNCTION GENERATE_ISSUES_FROM_DOC(doc STRING)
RETURNS ARRAY<STRUCT<description: STRING, email: STRING, priority: STRING, created_on:STRING>>
RETURN from_json(
    ai_query("databricks-dbrx-instruct",  doc),
    "array<struct<description:string, email:string, priority:string, created_on:string>>"
);

In [0]:
import pyspark.sql.functions as F

tickets = (
    df_with_prompt.sample(0.5)
    .selectExpr('id as doc_id', 'prompt')
    .withColumn("ticket", F.expr("GENERATE_ISSUES_FROM_DOC(prompt)"))
)

tickets = tickets.drop('prompt')
tickets = tickets.withColumn('ticket', F.explode('ticket'))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customer_tickets (
   id BIGINT GENERATED ALWAYS AS IDENTITY
)

In [0]:
tickets.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("customer_tickets")